[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/04_Opset_and_Metadata/Opset_and_Metadata_Deep_Dive.ipynb)

# 1.4 Opset and Metadata — Deep Dive

Understand **OpSet versioning** — the "language version" that controls which operator definitions your model uses — and how to annotate models with **metadata** for traceability and governance.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [What is an OpSet?](#section-1) | Formal definition and motivation |
| 2 | [Version Resolution Algorithm](#section-2) | How the runtime selects operator versions |
| 3 | [OpSet Domains](#section-3) | Standard, ML, and custom domains |
| 4 | [Inspecting and Changing OpSets](#section-4) | Practical code examples |
| 5 | [Model Metadata Fields](#section-5) | Producer, version, doc_string, and custom properties |
| 6 | [Setting Metadata Programmatically](#section-6) | Code walkthrough |
| 7 | [Version Compatibility Constraints](#section-7) | IR version vs opset version |
| 8 | [Practical OpSet Selection Strategy](#section-8) | How to choose the right opset |
| 9 | [Exploring the Operator Registry](#section-9) | Querying available operators |
| 10 | [Key Takeaways & Interview Questions](#section-10) | Summary and self-test |

### Prerequisites

- Completed **1.1–1.3** (graph construction, serialization, initializers)
- Basic understanding of semantic versioning

<a id='section-1'></a>
## Section 1: What Is an OpSet?

### Definition 1.1 (Operator Set)

An **OpSet** (Operator Set) is a versioned collection of operator definitions. It specifies, for each operator name, the exact **schema** (inputs, outputs, attributes, type constraints) that the operator follows.

Formally, an opset at version $v$ is:

$$\mathcal{O}_v = \{(\text{name}_i, \text{schema}_i, \text{since}_i) \mid i = 1, \ldots, |\mathcal{O}_v|\}$$

where $\text{since}_i \leq v$ is the version at which operator $i$ was last updated.

### Why Versioning?

Operators evolve over time — new attributes are added, type constraints are relaxed, semantics are refined. Without versioning, a model exported today might not run correctly on a runtime from six months ago (or vice versa).

The opset version creates a **contract** between the model and the runtime:

$$\text{Model says: } \text{"I use opset } v\text{"} \quad \Longrightarrow \quad \text{Runtime must support opset } v$$

### Version Resolution Rule

**Key Rule**: For a graph with opset version $v$, each operator uses the **most recent schema version that does not exceed** $v$:

$$\text{schema}(\text{op}, v) = \text{schema}_{v'} \quad \text{where } v' = \max\{s \mid s \leq v, \text{op defined at version } s\}$$

### Example: The `Add` Operator

```
Add operator version history:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
since_version=1:  Basic element-wise addition
since_version=6:  Axis/broadcast attributes removed
since_version=7:  Multidirectional broadcasting
since_version=13: BFloat16 support added
since_version=14: No change (latest)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

If graph opset = 15:
  → Add uses version 14 schema (max ≤ 15)

If graph opset = 10:
  → Add uses version 7 schema (max ≤ 10)

If graph opset = 5:
  → Add uses version 1 schema (max ≤ 5)
```

### Backward Compatibility Guarantee

$$\text{valid}(\mathcal{M}, \mathcal{O}_v) \implies \text{valid}(\mathcal{M}, \mathcal{O}_{v'}) \quad \forall v' \geq v$$

A model exported at opset $v$ will remain valid at any higher opset version. Operators are never removed, only extended.

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install onnx onnxruntime matplotlib numpy

In [ ]:
import numpy as np
import onnx
from onnx import TensorProto, defs
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid, set_model_props)
from onnx.checker import check_model
import onnxruntime as ort

print(f'ONNX version: {onnx.__version__}')
print(f'IR version:   {onnx.IR_VERSION}')
print(f'ONNX Runtime: {ort.__version__}')

<a id='section-2'></a>
## Section 2: Version Resolution Algorithm

### The Resolution Process

When a runtime loads an ONNX model, it resolves each operator's schema through this algorithm:

```
RESOLVE_OPERATOR(op_type, domain, graph_opset_version):
┌─────────────────────────────────────────────────────────┐
│  1. Find all schema versions for (op_type, domain)      │
│     → versions = {v₁, v₂, ..., vₖ}                     │
│                                                         │
│  2. Filter: keep only v ≤ graph_opset_version           │
│     → candidates = {vᵢ | vᵢ ≤ graph_opset}             │
│                                                         │
│  3. Select: schema_version = max(candidates)            │
│                                                         │
│  4. If candidates is empty → ERROR (op not available)   │
└─────────────────────────────────────────────────────────┘
```

### Formal Definition

Let $\mathcal{V}(\text{op})$ be the set of all versions at which operator "op" has a schema definition. The resolved version is:

$$v^*(\text{op}, v_{\text{graph}}) = \max\{v \in \mathcal{V}(\text{op}) \mid v \leq v_{\text{graph}}\}$$

If $\{v \in \mathcal{V}(\text{op}) \mid v \leq v_{\text{graph}}\} = \emptyset$, the operator is unavailable at this opset.

In [ ]:
# Demonstrate version resolution for common operators
operators = ['Add', 'MatMul', 'Relu', 'Softmax', 'Conv', 'BatchNormalization',
             'Reshape', 'Transpose', 'Gather', 'Where']

print(f'{"Operator":>20s} | {"Version History (since_version)"}')
print('-' * 70)

for op_name in operators:
    schemas = [s for s in defs.get_all_schemas_with_history()
               if s.name == op_name and s.domain == '']
    versions = sorted(set(s.since_version for s in schemas))
    print(f'{op_name:>20s} | {versions}')

# Show resolution for different graph opsets
print(f'\nResolution example for Softmax:')
softmax_versions = sorted(set(s.since_version for s in defs.get_all_schemas_with_history()
                              if s.name == 'Softmax' and s.domain == ''))
for graph_opset in [7, 11, 13, 15, 18, 21]:
    candidates = [v for v in softmax_versions if v <= graph_opset]
    resolved = max(candidates) if candidates else 'N/A'
    print(f'  graph_opset={graph_opset:2d} → Softmax uses schema v{resolved}')

<a id='section-3'></a>
## Section 3: OpSet Domains

### What Are Domains?

ONNX supports **multiple operator domains**, each with its own independent version numbering. A model can import operators from several domains simultaneously.

### Standard Domains

| Domain String | Full Name | Description | Example Operators |
|:---|:---|:---|:---|
| `""` (empty) | `ai.onnx` | Core neural network ops | MatMul, Conv, Relu, Softmax |
| `"ai.onnx.ml"` | ONNX-ML | Classical ML ops | TreeEnsembleRegressor, SVMClassifier |
| `"ai.onnx.training"` | Training | Gradient computation | Gradient, Adagrad, Momentum |

### Custom Domains

You can define **custom operator domains** for vendor-specific or research operators:

```python
opset_imports = [
    make_opsetid('', 18),           # standard ONNX ops
    make_opsetid('ai.onnx.ml', 3),  # ML-specific ops
    make_opsetid('my.company', 1),  # custom ops
]
```

### Domain Architecture

```
┌─────────────────────────────────────────────────────────┐
│                   ModelProto                            │
│                                                         │
│  opset_import:                                          │
│  ┌─────────────────────────────────────┐               │
│  │ domain=""        version=18         │ ← ai.onnx     │
│  │ domain="ai.onnx.ml" version=3       │ ← ML ops      │
│  │ domain="custom"  version=1          │ ← custom ops  │
│  └─────────────────────────────────────┘               │
│                                                         │
│  graph:                                                 │
│  ┌─────────────────────────────────────┐               │
│  │ Node(MatMul, domain="")             │ ← uses v18    │
│  │ Node(Add, domain="")               │ ← uses v18    │
│  │ Node(TreeEnsemble, domain="ml")     │ ← uses v3     │
│  │ Node(MyOp, domain="custom")        │ ← uses v1     │
│  └─────────────────────────────────────┘               │
└─────────────────────────────────────────────────────────┘
```

In [ ]:
# Build a model and set opset versions for multiple domains
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

graph = make_graph(
    [make_node('MatMul', ['X', 'A'], ['XA']),
     make_node('Add', ['XA', 'B'], ['Y'])],
    'lr', [X, A, B], [Y])

# Method 1: Set opset at model creation
model = make_model(graph, opset_imports=[
    make_opsetid('', 18),
    make_opsetid('ai.onnx.ml', 3)
])

print('Opset imports (set at creation):')
for op in model.opset_import:
    domain = op.domain or 'ai.onnx (default)'
    print(f'  domain={domain!r:25s}  version={op.version}')

# Method 2: Modify opset after creation
del model.opset_import[:]
op1 = model.opset_import.add()
op1.domain = ''
op1.version = 15

op2 = model.opset_import.add()
op2.domain = 'ai.onnx.ml'
op2.version = 3

check_model(model)
print('\nOpset imports (modified post-creation):')
for op in model.opset_import:
    domain = op.domain or 'ai.onnx (default)'
    print(f'  domain={domain!r:25s}  version={op.version}')

<a id='section-4'></a>
## Section 4: Inspecting and Changing OpSets

### Reading Model OpSet Information

Every `ModelProto` stores its opset imports in the `opset_import` repeated field. This is the first thing a runtime reads to determine how to interpret the graph's operators.

In [ ]:
# Build a default model and inspect all metadata fields
model_default = make_model(graph)
check_model(model_default)

print('Complete Model Metadata')
print('=' * 60)

fields = [
    ('ir_version', model_default.ir_version),
    ('model_version', model_default.model_version),
    ('producer_name', model_default.producer_name or '(empty)'),
    ('producer_version', model_default.producer_version or '(empty)'),
    ('domain', model_default.domain or '(empty)'),
    ('doc_string', model_default.doc_string or '(empty)'),
]

for name, value in fields:
    print(f'  {name:20s}: {value}')

print(f'\n  opset_import:')
for op in model_default.opset_import:
    print(f'    domain={op.domain or "ai.onnx"!r:15s}  version={op.version}')

print(f'\n  metadata_props:')
for prop in model_default.metadata_props:
    print(f'    {prop.key} = {prop.value}')
if not model_default.metadata_props:
    print(f'    (none)')

In [ ]:
# Demonstrate changing opset and verifying the model still works
opset_versions_to_test = [11, 13, 15, 17, 18]

x = np.random.randn(3, 2).astype(np.float32)
a = np.random.randn(2, 1).astype(np.float32)
b = np.random.randn(1, 1).astype(np.float32)
expected = x @ a + b

print(f'{"OpSet":>6s} | {"Valid":>6s} | {"Result matches":>15s}')
print('-' * 40)

for v in opset_versions_to_test:
    test_model = make_model(graph, opset_imports=[make_opsetid('', v)])
    try:
        check_model(test_model)
        valid = True
    except Exception:
        valid = False

    if valid:
        sess = ort.InferenceSession(
            test_model.SerializeToString(),
            providers=['CPUExecutionProvider'])
        result = sess.run(None, {'X': x, 'A': a, 'B': b})[0]
        match = np.allclose(result, expected)
    else:
        match = 'N/A'

    print(f'{v:>6d} | {str(valid):>6s} | {str(match):>15s}')

<a id='section-5'></a>
## Section 5: Model Metadata Fields

### The ModelProto Metadata Schema

Every ONNX model carries metadata that supports traceability, governance, and debugging:

```
ModelProto metadata fields:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ir_version          : int64    ← IR spec version
model_version       : int64    ← your model's version
producer_name       : string   ← tool that created it
producer_version    : string   ← version of that tool
domain              : string   ← model domain namespace
doc_string          : string   ← human-readable description
metadata_props      : map      ← arbitrary key-value pairs
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

### Field Descriptions

| Field | Type | Purpose | Example |
|:---|:---|:---|:---|
| `ir_version` | `int64` | Which ONNX IR spec the model follows | `8`, `9` |
| `model_version` | `int64` | Your internal versioning for the model | `1`, `42` |
| `producer_name` | `string` | The framework or tool that created the model | `"pytorch"`, `"skl2onnx"` |
| `producer_version` | `string` | Version of that framework | `"2.1.0"` |
| `domain` | `string` | Namespace for model_version (like a Maven groupId) | `"com.mycompany.ml"` |
| `doc_string` | `string` | Free-text description | `"Image classifier for cats vs dogs"` |
| `metadata_props` | `map<string, string>` | Arbitrary key-value pairs for custom metadata | `{"author": "team-ml"}` |

### Why Metadata Matters

In production ML systems, models are **artifacts** that must be tracked, versioned, and audited:

```
┌──────────────────────────────────────────────────────────┐
│              MODEL GOVERNANCE PIPELINE                    │
│                                                          │
│  Training  ──▶  Export  ──▶  Registry  ──▶  Deployment   │
│                    │            │               │        │
│                    ▼            ▼               ▼        │
│              producer_name  model_version  metadata_props│
│              producer_ver   domain         (git_sha,     │
│              doc_string                     dataset,     │
│                                             accuracy)    │
└──────────────────────────────────────────────────────────┘
```

<a id='section-6'></a>
## Section 6: Setting Metadata Programmatically

Let's build a fully-annotated model with all metadata fields populated.

In [ ]:
from onnx.helper import set_model_props

# Build model
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

graph = make_graph(
    [make_node('MatMul', ['X', 'A'], ['XA']),
     make_node('Add', ['XA', 'B'], ['Y'])],
    'linear_regression', [X, A, B], [Y])

model = make_model(graph, opset_imports=[make_opsetid('', 18)])

# Set standard metadata
model.ir_version = 8
model.model_version = 3
model.producer_name = 'my_training_pipeline'
model.producer_version = '2.1.0'
model.domain = 'com.example.ml'
model.doc_string = 'Linear regression model for house price prediction. '\
                   'Trained on housing_v3 dataset with SGD optimizer.'

# Set custom metadata (key-value pairs)
custom_metadata = {
    'author': 'Data Science Team',
    'dataset': 'housing_v3',
    'training_date': '2024-01-15',
    'git_commit': 'abc123def',
    'accuracy_rmse': '0.0342',
    'license': 'Apache-2.0',
}
set_model_props(model, custom_metadata)

check_model(model)

# Display all metadata
print('Model Metadata Report')
print('=' * 60)
print(f'  ir_version:       {model.ir_version}')
print(f'  model_version:    {model.model_version}')
print(f'  producer_name:    {model.producer_name}')
print(f'  producer_version: {model.producer_version}')
print(f'  domain:           {model.domain}')
print(f'  doc_string:       {model.doc_string[:60]}...')
print(f'\n  opset_import:')
for op in model.opset_import:
    print(f'    {op.domain or "ai.onnx":15s} v{op.version}')
print(f'\n  metadata_props:')
for prop in model.metadata_props:
    print(f'    {prop.key:20s} = {prop.value}')

In [ ]:
# Verify metadata survives serialization round-trip
model_bytes = model.SerializeToString()
model_loaded = onnx.load_model_from_string(model_bytes)

print('Round-trip metadata verification:')
print(f'  producer_name matches:  {model.producer_name == model_loaded.producer_name}')
print(f'  model_version matches:  {model.model_version == model_loaded.model_version}')
print(f'  doc_string matches:     {model.doc_string == model_loaded.doc_string}')

orig_props = {p.key: p.value for p in model.metadata_props}
loaded_props = {p.key: p.value for p in model_loaded.metadata_props}
print(f'  metadata_props match:   {orig_props == loaded_props}')
print(f'  byte-identical:         {model_bytes == model_loaded.SerializeToString()}')

<a id='section-7'></a>
## Section 7: Version Compatibility Constraints

### Two Independent Version Axes

ONNX has **two** version numbers that are often confused:

| Version | Controls | Where Set | Changes |
|---------|----------|-----------|--------|
| **IR version** (`ir_version`) | Container format (how the protobuf is structured) | `ModelProto.ir_version` | Rarely (major structural changes) |
| **OpSet version** (`opset_import.version`) | Operator semantics (what each op does) | `ModelProto.opset_import` | Frequently (new ops, updated schemas) |

### Compatibility Matrix

$$\text{IR version} \times \text{OpSet version} \to \text{Compatibility}$$

```
IR version and OpSet version are orthogonal:

IR version 7  ──── supports opset 1–15
IR version 8  ──── supports opset 1–18
IR version 9  ──── supports opset 1–20+

Each IR version adds NEW structural features:
  IR 3: Training info support
  IR 4: External data support
  IR 7: Type/shape annotations on intermediate values
  IR 8: Sequence/Map/Optional types, functions
  IR 9: Enhanced function support
```

### The Constraint

The IR version must be **compatible** with the opset version. The ONNX checker enforces this. A common error is setting an opset that's too new for the IR version:

$$\text{ir\_version} \geq \text{min\_ir}(\text{opset\_version})$$

In [ ]:
# Demonstrate IR version compatibility
print('Testing IR version × OpSet version compatibility:')
print(f'{"IR ver":>7s} | {"OpSet":>6s} | {"Valid":>6s} | {"Notes"}')
print('-' * 55)

test_combos = [
    (7, 11, 'Common legacy combo'),
    (7, 15, 'Works: IR 7 supports opset 15'),
    (8, 15, 'Recommended minimum'),
    (8, 18, 'Modern default'),
    (9, 18, 'Latest IR'),
    (9, 20, 'Cutting edge'),
]

for ir_ver, opset_ver, notes in test_combos:
    test_model = make_model(graph, opset_imports=[make_opsetid('', opset_ver)])
    test_model.ir_version = ir_ver
    try:
        check_model(test_model)
        valid = 'Yes'
    except Exception as e:
        valid = 'No'
    print(f'{ir_ver:>7d} | {opset_ver:>6d} | {valid:>6s} | {notes}')

<a id='section-8'></a>
## Section 8: Practical OpSet Selection Strategy

### Decision Framework

Choosing the right opset version involves balancing **feature availability** against **runtime compatibility**:

```
┌────────────────────────────────────────────────────────────┐
│           OPSET SELECTION DECISION TREE                    │
│                                                            │
│  Q1: What operators does your model use?                   │
│  └─▶ Find the minimum opset that defines all of them      │
│                                                            │
│  Q2: What runtime will you deploy on?                      │
│  └─▶ Check the maximum opset it supports                  │
│                                                            │
│  Q3: Do you need specific features?                        │
│  └─▶ e.g., opset 13 added BFloat16 for some ops           │
│       opset 18 added GroupNormalization                     │
│                                                            │
│  ANSWER: opset = max(min_required, min_for_features)       │
│          BUT  ≤ runtime_max_supported                      │
└────────────────────────────────────────────────────────────┘
```

### Common Opset Choices

| OpSet | Why Choose It | Key Features |
|:-----:|:---|:---|
| 11 | Maximum compatibility with older runtimes | Basic ops, dynamic shapes |
| 13 | Good balance of features + compatibility | BFloat16, improved Squeeze/Unsqueeze |
| 15 | Recommended default for most projects | Shape inference improvements |
| 17 | Modern features | LayerNorm, new reduction ops |
| 18+ | Cutting edge | GroupNorm, latest additions |

### Rule of Thumb

$$\text{opset} = \min\{v \mid v \geq v_{\text{min\_required}} \text{ AND } v \leq v_{\text{runtime\_max}}\}$$

When in doubt, use **opset 15** — it's widely supported and covers the vast majority of operations.

<a id='section-9'></a>
## Section 9: Exploring the Operator Registry

The ONNX Python package includes a complete registry of all operator schemas. Let's explore it programmatically.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Get all schemas from the default domain
all_schemas = defs.get_all_schemas_with_history()
default_schemas = [s for s in all_schemas if s.domain == '']
ml_schemas = [s for s in all_schemas if s.domain == 'ai.onnx.ml']

# Latest version of each operator
latest_ops = {}
for s in default_schemas:
    if s.name not in latest_ops or s.since_version > latest_ops[s.name].since_version:
        latest_ops[s.name] = s

latest_ml_ops = {}
for s in ml_schemas:
    if s.name not in latest_ml_ops or s.since_version > latest_ml_ops[s.name].since_version:
        latest_ml_ops[s.name] = s

print(f'Operator Registry Summary')
print(f'=' * 50)
print(f'Default domain (ai.onnx):')
print(f'  Total unique operators:     {len(latest_ops)}')
print(f'  Total schemas (all vers):   {len(default_schemas)}')
print(f'ML domain (ai.onnx.ml):')
print(f'  Total unique operators:     {len(latest_ml_ops)}')
print(f'  Total schemas (all vers):   {len(ml_schemas)}')

# Count operators introduced per opset version
intro_counts = Counter()
first_appearance = {}
for s in default_schemas:
    if s.name not in first_appearance:
        first_appearance[s.name] = s.since_version
    else:
        first_appearance[s.name] = min(first_appearance[s.name], s.since_version)

for name, ver in first_appearance.items():
    intro_counts[ver] += 1

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

versions = sorted(intro_counts.keys())
counts = [intro_counts[v] for v in versions]
cumulative = np.cumsum(counts)

ax1.bar(versions, counts, color='steelblue', edgecolor='navy', alpha=0.8)
ax1.set_xlabel('OpSet Version', fontsize=11)
ax1.set_ylabel('New Operators Introduced', fontsize=11)
ax1.set_title('Operator Introduction by OpSet Version', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

ax2.plot(versions, cumulative, 'o-', color='#E74C3C', linewidth=2, markersize=6)
ax2.fill_between(versions, 0, cumulative, alpha=0.1, color='red')
ax2.set_xlabel('OpSet Version', fontsize=11)
ax2.set_ylabel('Cumulative Operators Available', fontsize=11)
ax2.set_title('Cumulative Operator Coverage', fontsize=12, fontweight='bold')
ax2.grid(alpha=0.3)

plt.suptitle('ONNX Operator Registry Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Inspect a specific operator's schema
def inspect_op(op_name, domain=''):
    schema = defs.get_schema(op_name, domain=domain)
    print(f'Operator: {op_name}')
    print(f'  Domain:        {schema.domain or "ai.onnx"}')
    print(f'  Since version: {schema.since_version}')
    print(f'  Doc (first 100 chars): {schema.doc[:100]}...')
    print(f'  Inputs:')
    for inp in schema.inputs:
        print(f'    {inp.name}: {inp.description[:60]}')
    print(f'  Outputs:')
    for out in schema.outputs:
        print(f'    {out.name}: {out.description[:60]}')
    if schema.attributes:
        print(f'  Attributes:')
        for name, attr in schema.attributes.items():
            print(f'    {name}: type={attr.type}, required={attr.required}')
    print()

inspect_op('MatMul')
inspect_op('Transpose')
inspect_op('Softmax')

<a id='section-10'></a>
## Section 10: Key Takeaways & Interview Questions

### Summary

| Concept | Key Point |
|---------|----------|
| **OpSet** | A versioned collection of operator definitions that establishes the semantic contract between model and runtime |
| **Resolution** | Each operator uses the most recent schema version $\leq$ the graph opset version |
| **Domains** | Multiple independent operator sets: default (`""`), ML (`"ai.onnx.ml"`), custom |
| **IR version** | Controls the container format (how protobuf is structured), independent of opset |
| **Metadata** | `producer_name`, `model_version`, `doc_string`, `metadata_props` for governance |

### Critical Rules

1. **Backward compatible**: Models exported at opset $v$ remain valid at any opset $v' \geq v$
2. **Not forward compatible**: A runtime supporting opset 15 **cannot** run a model requiring opset 18 features
3. **Use `make_opsetid()`** to set opsets cleanly; manual protobuf manipulation is error-prone
4. **IR version matters** for structural features (functions, external data); opset version matters for operator semantics

### Interview Questions

1. **Q**: What is the difference between IR version and OpSet version?
   - **A**: IR version controls the model container format (protobuf structure, supported features like functions or external data). OpSet version controls the semantics of individual operators. They are independent version axes.

2. **Q**: If your model uses opset 15 and a runtime supports opset 18, will it work?
   - **A**: Yes. ONNX guarantees backward compatibility — models at opset $v$ remain valid at any $v' \geq v$.

3. **Q**: How does ONNX resolve which version of an operator to use?
   - **A**: It selects the most recent schema version that does not exceed the graph's opset version: $v^* = \max\{v \in \mathcal{V}(\text{op}) \mid v \leq v_{\text{graph}}\}$

4. **Q**: Why would you use `metadata_props` on a production model?
   - **A**: For traceability and governance — recording training dataset, git commit, accuracy metrics, author, and license directly in the model file. This metadata survives serialization and can be queried by model registries.

---

**Next:** [Subgraphs, Tests, and Loops](../05_Subgraphs_Tests_Loops/) — ONNX control flow operators.